# B01 — land ads for layer 3 (MVP circles)
Task `tasks/T-008-b01-land-ads.md` (brief B-019). Rules only (regex + SQL), no ML/LLM model.

**Sources** (queried remotely at the pins in `config.py`; needs internet)
- `RevX_OCR_results` `block_level` (`config.DATASETS["ocr_blocks"]`): OCR text blocks, `collection = 'chronicling_america'`.
- `loc_chronicling_america_1770-1810` (`config.DATASETS["news"]`): paper title, place of publication, page image links (IIIF).
- `data/processed/places.parquet` (A04): county and state centroids with first/last dates (Newberry AHCB).

**Grain**
- `data/interim/ads.parquet`: one row per land-sale **block** (one printing of an ad in one issue), with its `run_id`.
- An **ad run** = the same ad text in the same paper across issues (repeats collapsed).
- `web/data/ads_places.json`: place × decade → `n_ads` (printings), `n_runs` (distinct ads), papers.

**Caveats up front** (`docs/DATA.md`, S3)
- The land-sale rule is a keyword filter, not a classifier: some rentals, sheriff sales of goods, and long multi-notice blocks get in.
- Papers skew to VA, DC and PA and to 1800–1810; 1810 is a single year.
- The place is the first county (or state) the ad names; when it names none, the ad sits at the paper's own county and is flagged.
- `usd_per_acre` is left empty for every ad until the historian's currency check (B-021) returns (PLAN §3a).

In [1]:
import sys, re, json, time, hashlib
from pathlib import Path
PIPELINE_DIR = Path.cwd().parent
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))
import config
from rx import io, remote
import duckdb, pandas as pd
pd.set_option("display.width", 220); pd.set_option("display.max_colwidth", 160); pd.set_option("display.max_rows", 80)

con = remote.connect()
con.sql("set http_retries=12; set http_retry_wait_ms=3000; set http_retry_backoff=2")
def q(sql): return con.sql(sql).df()
WEB_DATA = config.PROJECT_DIR / "web" / "data"
for k in ("ocr_blocks", "news"):
    print(k, config.DATASETS[k]["revision"][:10], remote.hf_url(k))
print("duckdb", duckdb.__version__)

ocr_blocks 92477706d1 hf://datasets/RevolutionCrossroads/RevX_OCR_results@92477706d19db4dcd0c425e01e1ff85a209a8269/block_level/train-*.parquet
news 0b97906ad3 hf://datasets/RevolutionCrossroads/loc_chronicling_america_1770-1810@0b97906ad32e269376872934820357fc333151d8/data/train-*.parquet
duckdb 2.0.0


## 1. Land-sale blocks
Rule from S2 / S3 (unchanged): lower-cased `block_text` matches `(\d+ acres | tract of land | lands? for sale)` **and**
`(for sale | to be sold | will be sold | at private sale | public sale)`. Only newspaper files (`filename` starts with `sn`).
`filename` = `<lccn>_<issue date>_<edition>.pdf`; `page_number` counts from 0 (page = `page_number + 1`); `block_bbox` is raw pixels in the OCR's page render.

In [2]:
RE_A = r"\d+ acres|tract of land|lands? for sale"
RE_B = r"for sale|to be sold|will be sold|at private sale|public sale"
t = time.time()
blocks = q(f'''
  select filename, page_number, block_id, block_type, block_bbox, block_text
  from read_parquet('{remote.hf_url("ocr_blocks")}')
  where collection = 'chronicling_america' and filename like 'sn%'
    and regexp_matches(lower(block_text), '{RE_A}') and regexp_matches(lower(block_text), '{RE_B}')''')
print(len(blocks), "land-sale blocks |", round(time.time() - t, 1), "s")

pages = q(f'''
  select lccn, newspaper_title, place_of_publication, issue_date::date as issue_date, edition_order, Page,
         Web_URL, thumbnail_url
  from read_parquet('{remote.hf_url("news")}')''')
papers = (pages.groupby("lccn").agg(paper_title=("newspaper_title", "first"),
                                    place_of_publication=("place_of_publication", "first")).reset_index())
print(len(pages), "pages |", len(papers), "papers")

26117 land-sale blocks | 24.5 s


77793 pages | 110 papers


In [3]:
f = blocks.filename.str.removesuffix(".pdf").str.split("_", expand=True)
blocks["lccn"] = f[0]
blocks["issue_date"] = pd.to_datetime(f[1]).astype("datetime64[us]")
blocks["edition_order"] = f[2].astype(int)
blocks["page"] = blocks.page_number + 1
blocks["year"] = blocks.issue_date.dt.year.astype(int)
blocks["decade"] = blocks.year // 10 * 10

pages["issue_date"] = pd.to_datetime(pages.issue_date).astype("datetime64[us]")
pg = pages[["lccn", "issue_date", "edition_order", "Page", "Web_URL", "thumbnail_url"]].rename(columns={"Page": "page"})
ads = (blocks.merge(papers, on="lccn", how="left", validate="m:1")
             .merge(pg, on=["lccn", "issue_date", "edition_order", "page"], how="left", validate="m:1"))
assert len(ads) == len(blocks), "join duplicated rows"
print("blocks without a page row in the news table:", ads.Web_URL.isna().sum())
ads.groupby("decade").agg(blocks=("block_id", "size"), papers=("lccn", "nunique")).T

blocks without a page row in the news table: 2


decade,1770,1780,1790,1800,1810
blocks,3362,2567,3511,14557,2120
papers,25,25,22,42,11


## 2. Ad runs (repeats collapsed)
Ads ran in many issues. Run key = paper (`lccn`) + the first 150 letters/digits of the lower-cased text (the S3 distinct-text key, now within one paper).
It still splits a reprint whose first lines changed and can merge two ads that open with the same boilerplate, so `n_runs` is an estimate.

In [4]:
key = ads.block_text.str.lower().str[:300].str.replace(r"[^a-z0-9]", "", regex=True).str[:150]
ads["run_id"] = [hashlib.md5(f"{l}|{k}".encode()).hexdigest()[:12] for l, k in zip(ads.lccn, key)]
ads = ads.sort_values(["run_id", "issue_date", "page", "block_id"]).reset_index(drop=True)
run = ads.groupby("run_id").agg(run_printings=("block_id", "size"), run_first=("issue_date", "min"), run_last=("issue_date", "max"))
ads = ads.join(run, on="run_id")
ads["is_run_first"] = ~ads.duplicated("run_id")
print("blocks", len(ads), "| runs", ads.run_id.nunique())
run.run_printings.describe(percentiles=[.5, .75, .9, .99]).round(1).to_frame().T

blocks 26117 | runs 12062


,count,mean,std,min,50%,75%,90%,99%,max
run_printings,12062.0,2.2,5.0,1.0,1.0,2.0,4.0,18.0,239.0


## 3. Place named in the ad
`places` holds counties (A04 names, with first/last dates) and states. Text and county names are normalised the same way
(lower case, letters only, "Saint" → "St", a trailing *e*/*s*/*es* dropped so "Clarke"/"Clark" and "Prince George's"/"Prince George" meet; line-break hyphens joined).

**County mentions:** `<1–3 words> county` and `county of <1–3 words>`, longest name first; "said/same/the county" is skipped.
A small alias list covers OCR and spelling variants seen in the blocks and two NY renames (Tryon → Montgomery 1784, Charlotte → Washington 1784).

**Choosing one county** among same-named counties that existed on the issue date (A05 style, state-aware and dated):
1. a state named right after the county ("Bourbon county, Kentucky");
2. the paper's own state;
3. a state named anywhere in the ad;
4. a neighbouring state of the paper, nearest first (e.g. DC → MD, then VA);
5. the only county of that name in existence on that date.
If no county of that name existed on the date, only steps 1–2 are tried (`date_ok = False`).

**State only:** when no county resolves, a state named in a phrase like "state of Ohio", "in Kentucky", "lands in Georgia" (rivers excluded: "on the Ohio").
A state other than the paper's own wins.
**Fallback:** nothing named → the paper's own county (`place_method = 'paper_place'`, `is_fallback = True`).
**Per run:** every printing of a run gets the run's place: the most frequent text-matched place among its printings, else the fallback.

In [5]:
pl = io.read_stage("processed", "places")

def keynorm(s):
    s = s.lower().replace("'", "").replace("’", "")
    s = re.sub(r"[^a-z]+", " ", s)
    s = re.sub(r"\bsaint\b", "st", s)
    return re.sub(r"\s+", " ", s).strip()

def compact(k):
    return re.sub(r"(?<=[a-z]{3})(e|s|es)$", "", k)

def norm(s):
    # keep , ; : . ( ) as tokens so a match can't run across "Windsor, county of Middlesex"
    s = re.sub(r"-\s+", "", s.lower()).replace("'", "").replace("’", "").replace("&", " and ")
    s = re.sub(r"\bst\.", "st", s)
    s = re.sub(r"\bsaint\b", "st", s)
    s = re.sub(r"[^a-z,;:.()]+", " ", s)
    s = re.sub(r"\s*([,;:.()])\s*", r" \1 ", s)
    return re.sub(r"\s+", " ", s).strip()

cty = pl[pl.county.notna()].copy()
cty["key"] = (cty.county.str.replace(r"\(.*?\)", "", regex=True)
                 .str.replace(r"\s+-\s+dc part$|\s+dc$", "", case=False, regex=True)
                 .map(keynorm).str.replace(r"\s+(dist|p|parish)$", "", regex=True).map(compact))
by_key = {k: g for k, g in cty.groupby("key")}
ALIAS = {"amberst": "amherst", "kenawha": "kanawha", "kenhawa": "kanawha", "loudon": "loudoun",
         "newcastle": "new castle", "queen ann": "queen anne"}
for k, v in ALIAS.items():
    by_key[k] = by_key[compact(v)]
RENAMED = {"tryon": "nys_montgomery", "charlotte": "nys_washington"}   # NY names before 1784
for k, pid in RENAMED.items():
    by_key[k] = pd.concat([by_key.get(k, cty.iloc[:0]), cty[cty.place_id == pid]])
KEYS = set(by_key)

STATE_WORDS = {
    "massachusetts": "MA", "connecticut": "CT", "new york": "NY", "new jersey": "NJ", "pennsylvania": "PA",
    "virginia": "VA", "north carolina": "NC", "south carolina": "SC", "new hampshire": "NH", "vermont": "VT",
    "maryland": "MD", "delaware": "DE", "georgia": "GA", "rhode island": "RI", "district of maine": "ME",
    "maine": "ME", "kentucky": "KY", "tennessee": "TN", "ohio": "OH", "indiana": "IN", "illinois": "IL",
    "mississippi territory": "MS", "territory north west of the ohio": "Northwest Territory",
    "north western territory": "Northwest Territory", "northwestern territory": "Northwest Territory",
    "district of columbia": "DC"}
_NAMES = "|".join(sorted(STATE_WORDS, key=len, reverse=True))
NOT_STATE = r"(?! river| company| co\b| bank| gazette| argus| herald)"
STATE_STRICT = re.compile(r"(?:(?:state|commonwealth|district|territory) of |\bin (?:the )?(?:state of )?|lands? (?:in|of) )("
                          + _NAMES + r")\b" + NOT_STATE)
STATE_LOOSE = re.compile(r"\b(" + _NAMES + r")\b" + NOT_STATE)

def states_in(t, strict=True):
    out = []
    for m in (STATE_STRICT if strict else STATE_LOOSE).finditer(t):
        w = m.group(1)
        if w in ("ohio", "mississippi", "delaware") and re.search(r"(the|on|of) $", t[max(0, m.start(1) - 4):m.start(1)]):
            continue   # the river
        out.append((m.start(1), m.end(), STATE_WORDS[w]))
    return out

CTY_BEFORE = re.compile(r"((?:[a-z]+ ){1,3})county\b")
CTY_AFTER = re.compile(r"\bcounty of ((?:[a-z]+ ?){1,3})")
STOP = {"the", "said", "same", "this", "that", "in", "of", "and", "a", "an", "for", "our", "which", "whose", "each", "such"}

def county_mentions(t):
    out = []
    for m in CTY_BEFORE.finditer(t):
        w = m.group(1).split()
        for n in (3, 2, 1):
            k = compact(" ".join(w[-n:])) if len(w) >= n else None
            if k in KEYS and w[-n] not in STOP:
                out.append((m.start(), m.end(), k)); break
    for m in CTY_AFTER.finditer(t):
        w = m.group(1).split()
        for n in (3, 2, 1):
            k = compact(" ".join(w[:n])) if len(w) >= n else None
            if k in KEYS and w[0] not in STOP:
                out.append((m.start(), m.end(), k)); break
    return sorted(out)

print(len(cty), "counties ->", len(KEYS), "match keys")
print(county_mentions(norm("lying in Prince George's county, Maryland, the county of Fairfax, Bourbon county Kentucky; in the said county")))
print(states_in(norm("lands on the Ohio, in the state of Ohio, Kentucky river, in Kentucky")))

1875 counties -> 1074 match keys
[(6, 30, 'prince georg'), (48, 66, 'fairfax'), (68, 82, 'bourbon')]
[(63, 71, 'KY')]


In [6]:
# the paper's own county (place of publication -> county containing it in 1770-1810)
PAPER_PLACE = {
    "Richmond, Va.": "vas_henrico", "Alexandria [Va.]": "vas_arlington", "Washington City [D.C.]": "dcs_washington",
    "Philadelphia [Pa.]": "pas_philadelphia", "Philadelphia, Pa.": "pas_philadelphia",
    "Lancaster, Pa.;Philadelphia, Pa.": "pas_philadelphia",
    "Lexington [Ky.]": "kys_fayette", "[Lexington, Ky.]": "kys_fayette",
    "Williamsburg [Va.]": "vas_jamescity", "Williamsburg, Va.": "vas_jamescity", "Norfolk [Va.]": "vas_norfolk",
    "Lynchburg [Va.]": "vas_campbell", "Martinsburg, Va.": "wvs_berkeley", "Martinsburgh, Va. [W. Va.]": "wvs_berkeley",
    "Shepherdstown, Va. [W. Va.]": "wvs_berkeley", "Wheeling, Va. [W. Va.]": "wvs_ohio",
    "New York [N.Y.]": "nys_newyork", "New-York [N.Y.]": "nys_newyork", "Kingston [N.Y.]": "nys_ulster",
    "Portsmouth [N.H.]": "nhs_rockingham", "Portsmouth, N.H.": "nhs_rockingham", "[Portsmouth, N.H.]": "nhs_rockingham",
    "Exeter, N.H.": "nhs_rockingham", "Concord, N.H.": "nhs_rockingham", "Keene, N.H.": "nhs_cheshire",
    "Dover [N.H.]": "nhs_strafford", "Hanover, N.H.": "nhs_grafton",
    "Burlington [N.J.]": "njs_burlington", "Chatham, N.J.": "njs_morris",
    "Portland, Me.": "mes_cumberland", "Portland [Me.]": "mes_cumberland",
    "Newport, R.I.": "ris_newport", "Newport [R.I.]": "ris_newport", "Providence [R.I.]": "ris_providence",
    "Warren, R.I.": "ris_bristol", "Hartford [Conn.]": "cts_hartford", "Hartford, Conn.": "cts_hartford",
    "Charles-town [S.C.]": "scs_charlestondist", "[Charleston] S.C.": "scs_charlestondist",
    "Charlestown [S.C.]": "scs_charlestondist", "Charlestown [i.e. Charleston, S.C.]": "scs_charlestondist",
    "Charles-Town, S.C.": "scs_charlestondist",
    "Wilmington [Del.]": "des_newcastle", "Wilmington, Del.": "des_newcastle", "Dover, Del.": "des_kent",
    "Boston [Mass.]": "mas_suffolk", "Salem, Mass.": "mas_essex", "Newbury-port [Mass.]": "mas_essex"}
assert set(papers.place_of_publication) <= set(PAPER_PLACE), set(papers.place_of_publication) - set(PAPER_PLACE)
assert set(PAPER_PLACE.values()) <= set(pl.place_id)
NEIGH = {"VA": ["MD", "KY", "NC", "DC", "PA"], "DC": ["MD", "VA"], "MD": ["VA", "PA", "DE", "DC"], "PA": ["NJ", "NY", "MD", "DE"],
         "KY": ["VA", "OH", "TN"], "NY": ["NJ", "CT", "VT", "PA", "MA"], "NH": ["ME", "MA", "VT"], "ME": ["NH", "MA"],
         "RI": ["MA", "CT"], "CT": ["NY", "MA", "RI"], "NJ": ["NY", "PA", "DE"], "DE": ["MD", "PA", "NJ"],
         "SC": ["NC", "GA"], "MA": ["NH", "RI", "CT", "NY", "VT", "ME"]}
state_of = pl.set_index("place_id").state
ads["paper_place_id"] = ads.place_of_publication.map(PAPER_PLACE)
ads["paper_state"] = ads.paper_place_id.map(state_of)

def resolve(text, d, pst):
    t = norm(text)
    loose = states_in(t, strict=False)
    named = [s for _, _, s in loose]
    for start, end, k in county_mentions(t):
        g = by_key[k]
        dated = g[(g.first_date <= d) & (g.last_date >= d)]
        date_ok = len(dated) > 0
        c = dated if date_ok else g
        near = [s for p, _, s in loose if 0 <= p - end <= 25][:1]
        steps = ([("state_named_next", c[c.state.isin(near)]), ("paper_state", c[c.state == pst])]
                 + ([("state_in_ad", c[c.state.isin(named)])]
                    + [("neighbour_state", c[c.state == ns]) for ns in NEIGH.get(pst, [])]
                    + [("unique_name", c)] if date_ok else []))
        for method, h in steps:
            if len(h) == 1:
                return h.place_id.iloc[0], "county", "county_" + method, k, date_ok
    strict = [s for _, _, s in states_in(t)]
    other = [s for s in strict if s != pst] or strict
    if other:
        return other[0], "state", "state_named", None, True
    return None, None, None, None, None

t = time.time()
res = [resolve(x, pd.Timestamp(d), s) for x, d, s in zip(ads.block_text, ads.issue_date, ads.paper_state)]
m = pd.DataFrame(res, index=ads.index, columns=["text_place_id", "text_precision", "text_method", "county_key_found", "date_ok"])
ads = ads.join(m)
print(round(time.time() - t, 1), "s")
ads.text_method.value_counts(dropna=False).to_frame("blocks").assign(pct=lambda x: (100 * x.blocks / len(ads)).round(1))

18.4 s


,blocks,pct
text_method,,
NaN,11868,45.4
county_paper_state,9033,34.6
county_state_named_next,2380,9.1
state_named,1494,5.7
county_neighbour_state,787,3.0
county_state_in_ad,498,1.9
county_unique_name,57,0.2


In [7]:
# one place per run: most frequent text-matched place among its printings, else the paper's county
mt = ads[ads.text_place_id.notna()]
run_place = (mt.groupby(["run_id", "text_place_id", "text_precision", "text_method"]).size().rename("n").reset_index()
               .sort_values(["run_id", "n"], ascending=[True, False]).drop_duplicates("run_id")
               .set_index("run_id")[["text_place_id", "text_precision", "text_method"]]
               .rename(columns={"text_place_id": "place_id", "text_precision": "precision", "text_method": "place_method"}))
print("runs whose printings matched more than one place:", (mt.groupby("run_id").text_place_id.nunique() > 1).sum())
ads = ads.join(run_place, on="run_id")
ads["is_fallback"] = ads.place_id.isna()
ads.loc[ads.is_fallback, "place_id"] = ads.loc[ads.is_fallback, "paper_place_id"]
ads.loc[ads.is_fallback, "precision"] = "county"
ads.loc[ads.is_fallback, "place_method"] = "paper_place"

rate = pd.DataFrame({
    "blocks": [len(ads), (~ads.is_fallback).sum(), (ads.precision.eq("county") & ~ads.is_fallback).sum(), ads.precision.eq("state").sum(), ads.is_fallback.sum()],
    "runs": [ads.run_id.nunique(), ads[~ads.is_fallback].run_id.nunique(), ads[ads.precision.eq("county") & ~ads.is_fallback].run_id.nunique(),
             ads[ads.precision.eq("state")].run_id.nunique(), ads[ads.is_fallback].run_id.nunique()]},
    index=["all", "place named in text (match)", "  county", "  state only", "fallback: paper's county"])
rate["pct_blocks"] = (100 * rate.blocks / len(ads)).round(1)
rate["pct_runs"] = (100 * rate.runs / ads.run_id.nunique()).round(1)
rate

runs whose printings matched more than one place: 40


,blocks,runs,pct_blocks,pct_runs
all,26117,12062,100.0,100.0
place named in text (match),14530,6963,55.6,57.7
county,12977,6336,49.7,52.5
state only,1553,627,5.9,5.2
fallback: paper's county,11587,5099,44.4,42.3


In [8]:
# match rate by decade and by paper state
by = lambda col: (ads.groupby(col).agg(blocks=("run_id", "size"), matched=("is_fallback", lambda s: int((~s).sum())))
                    .assign(pct_matched=lambda x: (100 * x.matched / x.blocks).round(1)))
display(by("decade").T)
by("paper_state").sort_values("blocks", ascending=False).T

decade,1770,1780,1790,1800,1810
blocks,3362.0,2567.0,3511.0,14557.0,2120.0
matched,2057.0,1316.0,1741.0,8131.0,1285.0
pct_matched,61.2,51.3,49.6,55.9,60.6


paper_state,VA,PA,DC,KY,NY,NH,NJ,ME,RI,CT,SC,DE,MA
blocks,13353.0,3774.0,2668.0,1725.0,1256.0,1167.0,568.0,557.0,500.0,269.0,161.0,81.0,38.0
matched,8101.0,2220.0,1473.0,851.0,748.0,284.0,336.0,184.0,188.0,89.0,15.0,33.0,8.0
pct_matched,60.7,58.8,55.2,49.3,59.6,24.3,59.2,33.0,37.6,33.1,9.3,40.7,21.1


## 4. Per-ad fields for later (acres, price)
- `acres`: the first number directly before "acre(s)" (commas removed); `acres_n` = how many acreage figures the block has (sheriff lists can have dozens).
- `price_as_stated`: the first currency mark attached to a number, as printed: `£ 5`, `$ 300`, `<number> pounds|shillings|dollars|pence|cents`,
  plus "per acre" wording if it follows (S3 regex; "pounds of / weight" removed first). Spelled-out prices ("six dollars") are missed.
- `currency_as_stated`: the unit in that match (`£`, `$`, `pounds`, …); `currency_qualifier`: first of sterling / specie / currency / current money / continental / proclamation in the ad.
- `per_acre_stated`: the price match carries per-acre wording.
- `usd_per_acre` stays empty (B-021 pending); `price_flag = 'pending_B-021'` where a price was found.

In [9]:
lt = ads.block_text.str.lower()
lt2 = lt.str.replace(r"pounds\s+(?:of|weight)", "", regex=True)
acre_all = lt.str.findall(r"(\d[\d,]*(?:\.\d+)?)\s*acres?\b")
ads["acres_n"] = acre_all.str.len().astype(int)
ads["acres"] = pd.to_numeric(acre_all.str[0].str.replace(",", "", regex=False), errors="coerce")

UNIT = r"(?:\s*(?:per|an|the|/)\s*acre)?"
MONEY = re.compile(r"((£|\$)\s?\d[\d,.]*" + UNIT + r"|\d[\d,.]*\s*(pounds|shillings|dollars|pence|cents|dols)\.?" + UNIT + ")")
mm = lt2.str.extract(MONEY)
ads["price_as_stated"] = mm[0].str.strip()
ads["currency_as_stated"] = mm[1].fillna(mm[2])
ads["per_acre_stated"] = ads.price_as_stated.str.contains("acre", na=False)
ads["currency_qualifier"] = lt.str.extract(r"\b(sterling|specie|current money|currency|continental|proclamation)\b")[0]
ads["usd_per_acre"] = pd.Series(float("nan"), index=ads.index, dtype="Float64")
ads["price_flag"] = ads.price_as_stated.notna().map({True: "pending_B-021", False: None})

pd.DataFrame({"blocks": [len(ads), ads.acres.notna().sum(), ads.price_as_stated.notna().sum(),
                         (ads.acres.notna() & ads.price_as_stated.notna()).sum(), ads.per_acre_stated.sum()]},
             index=["all", "with acres", "with price", "with both", "price per acre stated"])

,blocks
all,26117
with acres,18685
with price,975
with both,803
price per acre stated,57


In [10]:
ads.loc[ads.price_as_stated.notna(), ["currency_as_stated", "currency_qualifier"]].fillna("-").value_counts().head(15).to_frame("blocks")

blocks
currency_as_stated currency_qualifier        
dollars            -                      445
$                  -                      183
£                  -                      159
cents              -                       63
pounds             -                       42
dols               -                       22
£                  sterling                17
                   current money           11
shillings          -                        8
£                  specie                   7
dollars            currency                 5
dols               proclamation             3
£                  currency                 2
$                  specie                   2
dollars            sterling                 1

## 5. Save `data/interim/ads.parquet`
One row per land-sale block. Image links: `iiif_service` is the LoC IIIF image service of the page; `page_image_url` is the whole page (1,000 px wide).
A crop of the ad itself is **not** built: `block_bbox` is in the OCR's page render, which is about 0.7× the IIIF image and not a fixed ratio
across pages; scaling needs the page width/height from the VLM_OCR bucket (`.chunks.json`, `docs/DATA.md`).

In [11]:
ads["ad_id"] = (ads.lccn + "_" + ads.issue_date.dt.strftime("%Y-%m-%d") + "_ed" + ads.edition_order.astype(str)
                + "_p" + ads.page.astype(str) + "_" + ads.block_id.str.replace("/page/", "", regex=False).str.replace("/", "-", regex=False))
ads["iiif_service"] = ads.thumbnail_url.str.split("/full/").str[0]
ads["page_image_url"] = ads.iiif_service + "/full/1000,/0/default.jpg"
ads["issue_url"] = ads.Web_URL
ads["county"] = ads.place_id.map(pl.set_index("place_id").county)
ads["state"] = ads.place_id.map(state_of)

COLS = ["ad_id", "run_id", "is_run_first", "run_printings", "run_first", "run_last",
        "lccn", "paper_title", "place_of_publication", "issue_date", "year", "decade", "edition_order", "page",
        "block_id", "block_type", "block_bbox", "block_text",
        "place_id", "county", "state", "precision", "place_method", "is_fallback", "county_key_found", "date_ok",
        "text_place_id", "paper_place_id",
        "acres", "acres_n", "price_as_stated", "currency_as_stated", "currency_qualifier", "per_acre_stated",
        "usd_per_acre", "price_flag", "iiif_service", "page_image_url", "issue_url"]
out = ads[COLS].sort_values(["issue_date", "lccn", "page", "block_id"]).reset_index(drop=True)
path = io.write_stage(out, "interim", "ads")
print(path, out.shape)

/Users/larawashington/Documents/Parsons/MajorStudio1/Projects/revolutionary-xroads/data/interim/ads.parquet (26117, 39)


## 6. Place × decade → `web/data/ads_places.json`
- `n_ads` = printings (blocks) in that place and decade; `n_runs` = distinct ads, each counted once, in the place and decade of its **first** printing
  (so `n_runs` sums to the number of runs); `n_ads_fallback` = printings placed at the paper's county because the ad names no place.
- `papers` = titles of the papers that printed them. `precision` = county or state (state rows sit at the state centroid).
- Decades start at the year (1770 = 1770–79); 1810 holds one year.

In [12]:
g = out.groupby(["place_id", "decade"])
cells = g.agg(n_ads=("ad_id", "size"), n_ads_fallback=("is_fallback", "sum"),
              n_papers=("lccn", "nunique"), papers=("paper_title", lambda s: sorted(s.dropna().unique()))).reset_index()
first = out[out.is_run_first].groupby(["place_id", "decade"]).size().rename("n_runs")
cells = cells.join(first, on=["place_id", "decade"]).fillna({"n_runs": 0})
meta = pl.set_index("place_id")[["county", "state", "lat", "lon"]]
cells = cells.join(meta, on="place_id")
cells["precision"] = cells.county.isna().map({True: "state", False: "county"})
cells = cells[["place_id", "county", "state", "precision", "lat", "lon", "decade", "n_ads", "n_runs", "n_ads_fallback", "n_papers", "papers"]]
cells[["decade", "n_ads", "n_runs", "n_ads_fallback", "n_papers"]] = cells[["decade", "n_ads", "n_runs", "n_ads_fallback", "n_papers"]].astype(int)
cells = cells.sort_values(["decade", "n_ads"], ascending=[True, False]).reset_index(drop=True)
cells["county"] = cells.county.astype(object).where(cells.county.notna(), None)   # state rows: null county in JSON

doc = {
    "meta": {
        "layer": "L3 land ads (MVP): circles at the place the ad names, sized by number of ads",
        "grain": "place x decade",
        "fields": {"n_ads": "printings (land-sale blocks)", "n_runs": "distinct ads, counted in the place and decade of their first printing",
                   "n_ads_fallback": "printings with no place named, placed at the paper's county",
                   "precision": "county or state centroid (places.parquet, Newberry AHCB)"},
        "sources": {"ocr_blocks": f"{config.DATASETS['ocr_blocks']['repo']}@{config.DATASETS['ocr_blocks']['revision']}",
                    "news": f"{config.DATASETS['news']['repo']}@{config.DATASETS['news']['revision']}",
                    "places": "data/processed/places.parquet (A04)"},
        "years": [int(out.year.min()), int(out.year.max())],
        "totals": {"ads": int(len(out)), "runs": int(out.run_id.nunique()), "matched_in_text": int((~out.is_fallback).sum())},
        "notebook": "rx-pipeline/notebooks/B01_ads.ipynb", "task": "T-008"},
    "places": cells.to_dict(orient="records")}
WEB_DATA.mkdir(parents=True, exist_ok=True)
jpath = WEB_DATA / "ads_places.json"
jpath.write_text(json.dumps(doc, ensure_ascii=False, allow_nan=False, separators=(",", ":")))
print(jpath, len(cells), "rows |", round(jpath.stat().st_size / 1024), "KB")
cells.groupby("decade").agg(places=("place_id", "nunique"), n_ads=("n_ads", "sum"), n_runs=("n_runs", "sum"), fallback=("n_ads_fallback", "sum")).T

/Users/larawashington/Documents/Parsons/MajorStudio1/Projects/revolutionary-xroads/web/data/ads_places.json 627 rows | 157 KB


decade,1770,1780,1790,1800,1810
places,142,97,108,200,80
n_ads,3362,2567,3511,14557,2120
n_runs,2166,1315,1673,6048,860
fallback,1305,1251,1770,6426,835


## 7. Checks for Lara
### 7a. Top 20 places (all decades)

In [13]:
top = (cells.groupby(["place_id", "county", "state", "precision"], dropna=False)
             .agg(n_ads=("n_ads", "sum"), n_runs=("n_runs", "sum"), n_ads_fallback=("n_ads_fallback", "sum"))
             .reset_index().sort_values("n_ads", ascending=False).head(20).reset_index(drop=True))
top["pct_fallback"] = (100 * top.n_ads_fallback / top.n_ads).round(0)
top

,place_id,county,state,precision,n_ads,n_runs,n_ads_fallback,pct_fallback
0,vas_henrico,Henrico,VA,county,2176,919,1957,90.0
1,pas_philadelphia,Philadelphia,PA,county,1717,963,1554,91.0
2,vas_arlington,Alexandria,VA,county,1492,380,1434,96.0
3,dcs_washington,Washington (Ext),DC,county,1266,407,1195,94.0
4,kys_fayette,Fayette,KY,county,1076,413,874,81.0
5,nhs_rockingham,Rockingham,NH,county,911,634,874,96.0
6,vas_jamescity,James City,VA,county,876,534,771,88.0
7,wvs_berkeley,Berkeley,VA,county,764,228,439,57.0
8,vas_norfolk,Norfolk (Ext),VA,county,679,225,610,90.0
9,VA,NaN,VA,state,566,174,0,0.0


### 7b. Ten sample ads with their matched place (text-matched runs, first printing, ordered by hash: reproducible)

In [14]:
s = (out[out.is_run_first & ~out.is_fallback].assign(h=lambda d: d.ad_id.map(lambda x: hashlib.md5(x.encode()).hexdigest()))
        .sort_values("h").head(10))
for i, r in enumerate(s.itertuples()):
    txt = re.sub(r"\s+", " ", r.block_text)
    k = r.county_key_found
    pos = txt.lower().find(k.split()[0]) if isinstance(k, str) else -1
    snip = txt[max(0, pos - 160):pos + 200] if pos > 160 else txt[:360]
    print(f"[{i}] {r.paper_title} ({r.place_of_publication}) {r.issue_date:%Y-%m-%d} p{r.page} | run of {r.run_printings}")
    print(f"    -> {r.place_id} ({r.county or ''}, {r.state}) via {r.place_method}; acres={r.acres}; price={r.price_as_stated}")
    print(f"    {snip}\n")

[0] The Kentucky gazette (Lexington [Ky.]) 1802-05-07 p3 | run of 1
    -> kys_woodford (Woodford, KY) via county_paper_state; acres=100.0; price=nan
    ublic auction, on the 13th day of May next, at the court house in Lexington, all the interest which John Hollingsworth decd. possessed in 100 acres of LAND, in Woodford county, on Clear creek, on which is erected, a MERCHANT & SAW MILL. —Sold to satisfy Cornelius Beatty and John Poettle- thwait, their debt and costs — Subject to a claim which Charles Wilkins

[1] The Pennsylvania journal, or, Weekly advertiser (Philadelphia, Pa.) 1776-02-14 p3 | run of 1
    -> pas_bucks (Bucks, PA) via county_paper_state; acres=100.0; price=nan
    ract of one hundred and eighty-three acres and one hundred and forty four perches of land or thereabouts, situate in the township of Southampton, the county of Bucks, about seventeen miles from the city of Philadelphia, on the road leading therefrom to Wrightstown; there is on the premises a new stone dwel

### 7c. Two fallback samples (no place named)

In [15]:
for r in out[out.is_run_first & out.is_fallback].sample(2, random_state=1).itertuples():
    print(f"{r.paper_title} {r.issue_date:%Y-%m-%d} -> {r.place_id} (paper's county)\n    {re.sub(chr(92)+'s+', ' ', r.block_text)[:300]}\n")

Virginia Argus 1806-07-16 -> vas_henrico (paper's county)
    Valuable Property FOR SALE. THE subscriber being desirous to remove from where he now lives, offers for sale, a valuable TAN YARD, in the town of New Canton, well enclosed with paling eight feet high, and twenty four good Tan Vats, with a large framed bark house, sixty by twenty four feet, with a go

The Kentucky gazette 1795-07-04 -> kys_fayette (paper's county)
    To be Sold. THREE hundred acres of first rate land, about nine miles from Lexington, near the head of Jet- famine, on which is a good square log house, kitchen, smoke house, barn, stable and an excellent herse-mill, with two pair of stones known by the name of the Cove (Horse) Mill, a good Hemp Mill



## 8. Assertions

In [16]:
a = pd.read_parquet(path)
assert len(a) == len(blocks) == 26117, len(a)                       # same block count as S3
assert a.ad_id.is_unique
for c in ["ad_id", "run_id", "lccn", "issue_date", "page", "block_id", "place_id", "paper_title", "precision"]:
    assert a[c].notna().all(), f"null {c}"
assert set(a.place_id) <= set(pl.place_id)
assert a.year.between(1770, 1810).all()
assert a.is_run_first.sum() == a.run_id.nunique()
assert a.groupby("run_id").place_id.nunique().max() == 1, "a run has more than one place"
assert a.usd_per_acre.isna().all(), "usd_per_acre must wait for B-021"
assert a.price_as_stated.notna().sum() == a.price_flag.notna().sum()
j = json.loads(jpath.read_text())["places"]
jd = pd.DataFrame(j)
assert not jd.duplicated(["place_id", "decade"]).any()
assert jd.n_ads.sum() == len(a) and jd.n_runs.sum() == a.run_id.nunique() and jd.n_ads_fallback.sum() == a.is_fallback.sum()
assert jd[["lat", "lon"]].notna().all().all()
assert (jd.decade % 10 == 0).all() and jd.decade.between(1770, 1810).all()
print("ok |", len(a), "ads,", a.run_id.nunique(), "runs,", len(jd), "place x decade rows; matched in text",
      round(100 * (~a.is_fallback).mean(), 1), "% of printings")

ok | 26117 ads, 12062 runs, 627 place x decade rows; matched in text 55.6 % of printings
